# Notebook 01 — Fundamentos y extracción

Primer sub-bloque del Tema 04. Cubre **por qué existe el ETL**, **los tres pasos**, **ETL vs ELT**, **idempotencia**, el **setup del entorno Python**, y la **extracción desde Aurora** con `pd.read_sql`. Cierra con una mención de cómo lucen las fuentes en la realidad (CSV, JSON, S3, APIs, message queues).

Al terminar este notebook deberías poder leer cualquier tabla de tu Aurora en un DataFrame de pandas y entender por qué ese paso es solo el principio de un proceso más largo.

## ¿Qué es el ETL y por qué existe?

En el Tema 02 viste que un **data warehouse** vive **separado** del sistema transaccional, conectado al OLTP por un "puente" que extrae los datos, los reorganiza al modelo dimensional y los carga al destino analítico. Ese puente tiene nombre: **ETL** — *Extract, Transform, Load*.

El problema de fondo que resuelve el ETL es **integrar fuentes heterogéneas** hacia un único destino. En una empresa real los datos no viven en un solo lugar limpio:

- El sistema de ventas (`northwind_oltp`) está en PostgreSQL.
- El sistema de RRHH está en SQL Server.
- Los logs de la app móvil llegan como JSON a S3 cada hora.
- Los datos del proveedor de marketing salen de su API en formato CSV.

Cada fuente con su propio formato, su propia frecuencia de actualización, sus propios bugs. El ETL es el proceso periódico (diario, horario, continuo) que **toma todos esos pedazos, los limpia, los reorganiza, y los deja en el DWH listos para análisis**. Sin ETL no hay DWH; sin DWH no hay BI.

En este módulo trabajamos con una sola fuente (`northwind_oltp`) para mantener el foco en los conceptos. El patrón se escala a 10 o 100 fuentes con la misma idea.

## Los tres pasos: Extract, Transform, Load

| Fase | Qué hace | Herramienta típica en este módulo |
|---|---|---|
| **Extract** | Lee datos del origen y los trae al espacio de procesamiento | `pd.read_sql`, `pd.read_csv` |
| **Transform** | Limpia, tipa, normaliza, calcula columnas derivadas, aplana relaciones | pandas (`merge`, `astype`, `to_datetime`, etc.) |
| **Load** | Escribe el resultado en el destino analítico | `df.to_sql` |

Este notebook cubre solo la **E**. Los notebooks 02 y 03 cubren la **T**. El notebook 04 cubre la **L** y empaqueta todo en un script productivo.

## ETL vs ELT — ¿cuál usar?

Hay una variante más nueva del patrón: **ELT** (*Extract, Load, Transform*). Cambia el orden — primero cargas los datos crudos al destino y después los transformas **con SQL en el destino mismo**, no en Python.

| | ETL clásico | ELT moderno |
|---|---|---|
| **Orden** | E → T → L | E → L → T |
| **Dónde se transforma** | En memoria (Python, herramienta ETL) | En el destino (SQL contra el DWH) |
| **Cuándo aplica** | DWH tradicional con recursos modestos | DWH columnar masivo (Snowflake, BigQuery, Redshift) |
| **Ventaja** | Datos llegan al destino ya limpios | Aprovecha el poder de cómputo del destino; transforma a escala |
| **Desventaja** | Tu máquina debe poder con todo el dataset en memoria | Almacenas datos crudos sin filtrar (más espacio en el DWH) |

En este módulo usamos **ETL clásico** — Aurora PostgreSQL es un DWH tradicional, los datos de Northwind caben holgadamente en pandas, y aprendes los conceptos de transformación explícita. La lógica de fondo es la misma; lo que cambia entre ETL y ELT es **dónde corre cada paso**.

## Idempotencia y por qué importa

Una propiedad crítica de cualquier pipeline ETL: si lo corres **dos veces seguidas**, el resultado en el destino debe ser **el mismo** que correrlo una vez. Eso es **idempotencia**.

```
estado_inicial → correr ETL → estado_final
estado_inicial → correr ETL → correr ETL → MISMO estado_final  ✓ idempotente
estado_inicial → correr ETL → correr ETL → datos duplicados   ✗ no idempotente
```

¿Por qué importa? Porque los ETLs **fallan**. En producción, lo que va a pasar tarde o temprano:

- El proceso aborta a la mitad por timeout de red.
- La instancia se cae y al reiniciar se reintenta el job del día.
- Un humano reejecuta manualmente porque "el dashboard se ve raro".

Si el ETL es idempotente, **reintentar es seguro**: no rompe nada, no duplica datos, no requiere intervención manual para limpiar. Si no es idempotente, cada falla se convierte en un incidente que alguien tiene que investigar y reparar.

**Patrón:** cargar con `if_exists='replace'` (drop + create + insert) hace al pipeline trivialmente idempotente — cada corrida deja la tabla destino exactamente como debe estar, sin importar qué había antes. Lo veremos en el Notebook 04.

## Setup del entorno Python

Lo que necesitas instalado:

- **Python 3.10+** y **Jupyter Lab**.
- **`pandas`** — manipulación de DataFrames.
- **`sqlalchemy`** — capa de abstracción sobre el driver de PostgreSQL.
- **`psycopg2-binary`** — driver concreto de PostgreSQL que SQLAlchemy invoca por debajo.

La forma recomendada es **Miniconda** + un ambiente dedicado al módulo (`bi-unam`). La guía paso a paso está en [**`anexos/instalar_miniconda.md`**](../anexos/instalar_miniconda.md) — cubre instalación por sistema operativo, creación del ambiente, instalación de las cuatro librerías y lanzamiento de Jupyter Lab.

Si ya tienes tu propio Python configurado, instala las librerías con:

```bash
pip install pandas sqlalchemy psycopg2-binary jupyterlab
```

Verifica versiones:

In [ ]:
import pandas as pd
import sqlalchemy
import psycopg2

print(f"pandas      {pd.__version__}")
print(f"sqlalchemy  {sqlalchemy.__version__}")
print(f"psycopg2    {psycopg2.__version__.split()[0]}")

## Conexión a Aurora con SQLAlchemy

El objeto central para hablar con Aurora es el **`engine`** de SQLAlchemy — un manejador de pool de conexiones que pandas usa por debajo para sus operaciones `read_sql` / `to_sql`.

La URL de conexión sigue este formato:

```
postgresql+psycopg2://<usuario>:<password>@<host>:<puerto>/<base>
```

Donde:

- `postgresql+psycopg2` — dialecto + driver.
- `<usuario>` — `postgres` en el cluster que creaste en el Tema 01.
- `<password>` — el master password del cluster.
- `<host>` — el endpoint writer (`xxxxx.cluster-yyyyy.us-east-1.rds.amazonaws.com`).
- `<puerto>` — `5432`.
- `<base>` — `northwind`.

**Nunca pongas el password en el código.** El patrón profesional es leerlo de una **variable de entorno**:

```bash
# Antes de lanzar Jupyter, en tu terminal:
export AURORA_PASSWORD='tu-password-real'
```

Así el password vive solo en tu sesión, no en el archivo del notebook (que podrías commitear sin querer).

In [ ]:
import os
from sqlalchemy import create_engine, text

# Reemplaza el host con tu endpoint writer del Tema 01
AURORA_HOST = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = os.environ["AURORA_PASSWORD"]

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/northwind"
)

**Smoke test** — confirmar que el engine conecta y la base responde:

In [ ]:
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print(version)

Si ves algo como `PostgreSQL 17.x on aarch64-unknown-linux-gnu...`, la conexión funciona.

## Extracción desde Aurora con `pd.read_sql`

Con el engine listo, leer una tabla completa hacia un DataFrame es una sola línea:

```python
df = pd.read_sql("SELECT * FROM schema.tabla", engine)
```

pandas se conecta, ejecuta la query, trae los resultados, infiere tipos, y te devuelve un DataFrame listo para manipular. Lo veremos con `customers`, `orders` y `order_details` — las tres tablas que más vamos a usar.

In [ ]:
df_customers = pd.read_sql(
    "SELECT * FROM northwind_oltp.customers",
    engine,
)
print(f"shape: {df_customers.shape}")
df_customers.head()

In [ ]:
df_orders = pd.read_sql(
    "SELECT * FROM northwind_oltp.orders",
    engine,
)
print(f"shape: {df_orders.shape}")
df_orders.head()

In [ ]:
df_order_details = pd.read_sql(
    "SELECT * FROM northwind_oltp.order_details",
    engine,
)
print(f"shape: {df_order_details.shape}")
df_order_details.head()

Después de cada extracción, conviene revisar:

- `df.shape` — `(filas, columnas)`. Si es `(0, n)`, algo está mal.
- `df.head()` — primeras filas para sanity-check visual.
- `df.dtypes` — tipos que pandas infirió. Veremos en el Notebook 02 cómo corregir los que se equivoquen.

In [ ]:
df_order_details.dtypes

### Extraer solo lo que necesitas

`SELECT *` está bien para aprender, pero en producción extraes **solo las columnas que vas a usar**. Cada columna extra es ancho de banda gastado y memoria ocupada en tu máquina.

Ejemplo: para construir `dim_customer` solo necesitas algunas columnas:

In [ ]:
df_customers_min = pd.read_sql(
    """
    SELECT customer_id, company_name, contact_name, contact_title,
           city, region, postal_code, country
    FROM   northwind_oltp.customers
    """,
    engine,
)
print(f"shape: {df_customers_min.shape}")
df_customers_min.head(3)

También puedes **filtrar en SQL** (`WHERE`) y **agregar en SQL** (`GROUP BY`) cuando solo necesitas un subconjunto o un resumen — siempre es más eficiente que traer todo y filtrar en pandas.

In [ ]:
df_clientes_alemanes = pd.read_sql(
    "SELECT * FROM northwind_oltp.customers WHERE country = 'Germany'",
    engine,
)
print(f"clientes alemanes: {len(df_clientes_alemanes)}")

## Lectura por chunks

Northwind cabe holgadamente en memoria (3 200 filas en total). En la vida real las tablas pueden tener millones o cientos de millones de filas — leer todo de un jalón puede:

- Reventar la memoria de tu máquina.
- Mantener una conexión a la base abierta por demasiado tiempo (timeout, locks).

Para esos casos, `pd.read_sql` acepta el parámetro `chunksize`, que **devuelve un iterador de DataFrames** en vez de uno solo:

In [ ]:
total_filas = 0
for i, chunk in enumerate(
    pd.read_sql(
        "SELECT * FROM northwind_oltp.order_details",
        engine,
        chunksize=500,
    )
):
    total_filas += len(chunk)
    print(f"chunk {i}: {chunk.shape[0]} filas (acumulado: {total_filas})")

Cada chunk es un DataFrame normal que puedes procesar individualmente. Patrón típico: leer chunk → transformar → cargar al destino → liberar memoria del chunk → siguiente chunk. Así procesas datasets que **no caben** en memoria.

Para Northwind no necesitamos chunks — los notebooks siguientes leen las tablas completas. Pero la técnica vale la pena conocerla.

## Cómo lucen las fuentes en la realidad

En este módulo extraes de **un PostgreSQL accesible por red**. Es el caso más cómodo. En la realidad las fuentes de un ETL son mucho más variadas:

| Tipo de fuente | Patrón en pandas | Cuándo aparece |
|---|---|---|
| **CSV / TSV** locales o de URL | `pd.read_csv("archivo.csv")` | Exports de sistemas legacy, datasets públicos, planillas de Excel guardadas como CSV |
| **JSON / JSONL** | `pd.read_json("archivo.json")` | Logs de aplicaciones, exports de APIs, NoSQL |
| **Excel** | `pd.read_excel("archivo.xlsx")` | Reportes manuales de áreas no técnicas |
| **Parquet** | `pd.read_parquet("archivo.parquet")` | Pipelines de data lake — formato columnar comprimido |
| **Object storage (S3)** | `pd.read_csv("s3://bucket/...")` con `s3fs` instalado | Datasets en la nube; zona de aterrizaje típica antes del DWH |
| **APIs REST** | `requests.get(url).json()` → `pd.DataFrame(...)` | Datos de proveedores externos, integraciones SaaS |
| **Otras DBs** (MySQL, SQL Server, Oracle) | Igual que aquí, cambia la URL de SQLAlchemy | Sistemas de otra área de la empresa |
| **Message queues** (Kafka, SQS) | Consumo continuo, no batch — fuera de alcance del módulo | Pipelines de streaming en tiempo real |

La fase **Extract** del ETL es esencialmente *"poner los datos en un DataFrame"*. El resto del pipeline no cambia según la fuente — una vez que tienes el `df`, las transformaciones son las mismas.

## Cierre

Ya tienes los datos del OLTP en DataFrames de pandas — la fase **E** del ETL está completa. Lo que viste:

- Por qué existe el ETL y dónde encaja en una arquitectura de BI.
- Los tres pasos (E/T/L), la variante ELT, y la propiedad de idempotencia.
- Cómo conectar a Aurora desde Python con SQLAlchemy.
- Cómo extraer tablas completas, filtradas, o por chunks con `pd.read_sql`.

El siguiente notebook (**02 — Limpieza y perfilado**) toma estos DataFrames y resuelve el primer paso de la fase **T**: entender qué hay dentro y qué problemas tienen los datos antes de transformarlos.

---

<p align="center">
<a href="Readme.md">← Volver al índice del Tema 04</a> | <a href="02_limpieza_y_perfilado.ipynb">Siguiente: Notebook 02 — Limpieza y perfilado →</a>
</p>